# The same day, over the API

§13 exposes the workflow as HTTP. The API is "a thin layer: it accepts inputs,
starts jobs, reports status and returns results" — stage logic lives in the
modules of §5 and never in a request handler.

This notebook drives it in-process: no server, no Redis, no ports. The queue is
real Arq on `fakeredis`, so the job path is the production one.

In [ ]:
from httpx import ASGITransport, AsyncClient

from ddn.api.store import Store
from tests.api_harness import drain, make_app, make_pool

pool = make_pool()
store = Store()
client = AsyncClient(transport=ASGITransport(app=make_app(pool, store)),
                     base_url="http://ddn")
print("ready")

## §13.2's resources

Ten rows in §13.2's table. How many paths that is, the cell counts rather
than the sentence claims — it said twenty-three for as long as it took two
more resources to land. The OpenAPI document is the published form of §9.

In [ ]:
spec = make_app(pool, store).openapi()
for path in sorted(spec["paths"]):
    verbs = " ".join(sorted(v.upper() for v in spec["paths"][path]))
    print(f"  {verbs:<12}{path}")
operations = sum(len(verbs) for verbs in spec["paths"].values())
print(f"\n{len(spec['paths'])} paths, {operations} operations")

## Ingest (§5.1.1)

The upload file usually arrives before the bag. `202`, not `201`: §5.2 has to
reconcile, geocode, assemble and sort before any of these is routable, so what
the hub has taken on is work rather than a set of envelopes.

`Idempotency-Key` is required. A driver app that retries after a timeout must
not create the envelope twice.

In [ ]:
envelope = {
    "package_id": "PKG-1", "customer_id": "CUST-001", "recipient_id": "RCPT-1",
    "package_type": "finished", "mailbag_id": "BAG-0001", "status": "Requested",
    "lat": 9.94, "lon": -84.08, "coord_source": "actual",
    "geocode_confidence": "high", "facility_id": "HUB",
    "priority": 1500.0, "sla_date": "2026-09-20",
}
r = await client.post("/envelopes/batch", json={"envelopes": [envelope]},
                      headers={"Idempotency-Key": "upload-1"})
print(r.status_code, r.json())

Without the key it is refused, and the reason says why.

In [ ]:
r = await client.post("/envelopes/batch", json={"envelopes": [envelope]})
print(r.status_code, r.json()["detail"])

## A replay returns the original response

Same key, different body. The second call does not act; it is handed the first
call's answer, with a header saying so.

In [ ]:
r = await client.post("/envelopes/batch",
                      json={"envelopes": [dict(envelope, package_id="PKG-999")]},
                      headers={"Idempotency-Key": "upload-1"})
print(r.status_code, r.json(), "replay:", r.headers.get("Idempotent-Replay"))
print("PKG-999 was not created:", "PKG-999" not in store.envelopes)

## Lifecycle by events (§5.2.6)

Status is never set directly. A caller appends what happened and the state
machine validates it. §5.2.6 draws no `Requested -> Delivered`, so that is a
409 carrying the state the envelope is actually in.

In [ ]:
r = await client.post("/envelopes/PKG-1/events",
                      json={"event": "Delivered", "actor": "driver-7"},
                      headers={"Idempotency-Key": "bad-move"})
print(r.status_code)
print(" current_state:", r.json()["current_state"])
print(" detail:", r.json()["detail"])

In [ ]:
for step, key in (("Collected", "e1"), ("Received at hub", "e2")):
    r = await client.post("/envelopes/PKG-1/events",
                          json={"event": step, "actor": "hub-1"},
                          headers={"Idempotency-Key": key})
    print(r.status_code, r.json())

## Solver work is a job (§13.1)

Any endpoint that invokes the solver returns `202` with a job id. Never
`BackgroundTasks`: a process that dies takes in-process work with it and nobody
can ask what became of it.

Here we run the worker in burst mode to drain the queue; in the stack that is a
separate container.

In [ ]:
r = await client.post("/allocation/runs", json={
    "day": "2026-09-17",
    "pools": {"HUB": 1600, "D1": 850, "D2": 650},
    "bikes": [f"MOTO-{n:03d}" for n in range(1, 41)],
})
accepted = r.json()
print(r.status_code, accepted)

print("before the worker runs:", (await client.get(accepted["poll"])).json()["status"])
await drain(pool)
done = (await client.get(accepted["poll"])).json()
print("after:", done["status"])
print("targets:", done["result"]["targets"], "moves:", done["result"]["moves"])
print("§7.1 violations:", done["violations"])

## Line-haul milestones, per leg (§13.2)

A trip is several legs, and §13.2 spells its milestones per leg: "leg
departed, leg arrived". The leg is the whole point of the names — §7.1's
dispatch-order check has to know *which* van landed where, and a trip-level
"arrived" cannot say.

The plan below is the smallest one that has legs at all: one van, two depots,
one envelope each.

In [ ]:
plan = (await client.post("/linehaul/plans", json={
    "day": "2026-09-16",
    "facilities": [{"id": "D1", "route_release_time": 7 * 3600,
                    "transit_from_hub_min": 30},
                   {"id": "D2", "route_release_time": 7 * 3600,
                    "transit_from_hub_min": 45}],
    "envelopes": [{"package_id": "P1", "facility_id": "D1",
                   "expected_ready_at": 0},
                  {"package_id": "P2", "facility_id": "D2",
                   "expected_ready_at": 0}],
    "vans": [{"vehicle_id": "VAN-01"}],
    "unload_seconds": 1800,
})).json()
await drain(pool)

trips = (await client.get(plan["poll"])).json()["result"]["trips"]
legs = [leg for trip in trips for leg in trip["legs"]]
for offset, leg in enumerate(legs):
    aboard = sum(len(ids) for ids in leg["hub_loads"].values())
    print(f"  leg {offset}: {leg['from_facility']:<4}-> {leg['to_facility']:<4}"
          f"{aboard} aboard")

`leg` is an offset into that list. The handler does not resolve it against the
plan: CLAUDE.md's §13 rule is that handlers carry no routing logic, and looking
the plan up would make this endpoint answer "that job has not finished yet",
which is not a question about an event.

In [ ]:
for event, key in (("leg departed", "leg-0-out"), ("leg arrived", "leg-0-in")):
    r = await client.post(f"/linehaul/plans/{plan['job_id']}/events",
                          json={"event": event, "leg": 0, "actor": "driver-3"},
                          headers={"Idempotency-Key": key})
    print(r.status_code, r.json())

print()
for entry in store.audit_for(plan["job_id"]):
    print(f"  {entry.at:%H:%M:%S}  {entry.actor:<10}{entry.action}")

### What it refuses

Two bodies, two reasons. `departed` is the trip-level name this service
accepted before v0.16; it was kept for one release so a driver app on a slower
cycle than the document would not break on a document edit, and that window is
closed. `leg arrived` with no `leg` carries exactly what `arrived` carried,
which is what §5.3 outgrew — so it is refused too.

In [ ]:
for body in ({"event": "departed", "actor": "driver-3"},
             {"event": "leg arrived", "actor": "driver-3"}):
    r = await client.post(f"/linehaul/plans/{plan['job_id']}/events", json=body,
                          headers={"Idempotency-Key": f"no-{body['event']}"})
    print(r.status_code, r.json()["detail"])
    assert r.status_code == 422, "both of these are 422s, and the text says so"

## An override, and the audit it leaves (§8.2, §13.1)

A lock is stored on the envelope where §9.1 puts it, and a fresh run is
enqueued rather than the old one edited — a plan that changed after it was read
is worse than a new plan that says so.

In [ ]:
# §8.2 applies an override to a *plan*, so there has to be one first.
# Locking against an id that names no run is a 404 — it used to be a 202
# for a job that then died on `KeyError: 'facility'`, which told the
# caller the override had been accepted when nothing had happened.
flat = [[0, 180], [180, 0]]
run = (await client.post("/routes/runs", json={
    "facility": {"id": "HUB", "lat": 9.94, "lon": -84.05,
                 "shift_start": 25200, "shift_end": 54000},
    "day": "2026-09-17",
    "bikes": [{"vehicle_id": "MOTO-050", "type": "motorbike",
                "facility_id": "HUB", "role": "delivery",
                "capacity_envelopes": 35, "capacity_weight_g": 35_000,
                "shift_start": 25200, "shift_end": 54000}],
    "matrix": {"durations": flat, "distances": flat},
})).json()

r = await client.post(f"/routes/runs/{run['job_id']}/locks", json={
    "package_id": "PKG-1", "locked_vehicle_id": "MOTO-050", "actor": "ops-anna"})
print(r.status_code, "re-run queued:", r.json()["job_id"][:8] + "…")
print("locked_vehicle_id:", store.envelopes["PKG-1"]["locked_vehicle_id"])

for entry in store.audit_for("PKG-1"):
    print(f"  {entry.at:%H:%M:%S}  {entry.actor:<10}{entry.action}")

## Capabilities, measured rather than declared

§12 Q5 asked five questions about the solver. `docs/solver-capabilities.md`
answered them by running each one; the `no` settles §4.2 for two-stage.

In [ ]:
for name, value in (await client.get("/solver/capabilities")).json().items():
    print(f"  {name:<40}{value}")

In [ ]:
await client.aclose()